## Car Price Prediction using Regularized Regression

# Section1: Data Summary

Dataset: Car sales dataset (Size:157x16)

Target variable: Price of cars(Price_in_thousands)

Feature variables:Car specks (Horsepower, Engine_size, Curb_weight, Fuel_efficiency, Wheelbase, Fuel_capacity)


In [2]:
import sklearn
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score,root_mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV,LassoCV
from sklearn.preprocessing import PolynomialFeatures

In [3]:
filepath = '/Users/antonistsihlakis/Documents/imb_ml/course2/Car_sales.csv'
data = pd.read_csv(filepath)
data.head()

,Manufacturer,Model,Sales_in_thousands,__year_resale_value,Vehicle_type,Price_in_thousands,Engine_size,Horsepower,Wheelbase,Width,Length,Curb_weight,Fuel_capacity,Fuel_efficiency,Latest_Launch,Power_perf_factor
0,Acura,Integra,16.919,16.360,Passenger,21.50,1.8,140.0,101.2,67.3,172.4,2.639,13.2,28.0,2/2/2012,58.280150
1,Acura,TL,39.384,19.875,Passenger,28.40,3.2,225.0,108.1,70.3,192.9,3.517,17.2,25.0,6/3/2011,91.370778
2,Acura,CL,14.114,18.225,Passenger,NaN,3.2,225.0,106.9,70.6,192.0,3.470,17.2,26.0,1/4/2012,NaN
3,Acura,RL,8.588,29.725,Passenger,42.00,3.5,210.0,114.6,71.4,196.6,3.850,18.0,22.0,3/10/2011,91.389779
4,Audi,A4,20.397,22.255,Passenger,23.99,1.8,150.0,102.6,68.2,178.0,2.998,16.4,27.0,10/8/2011,62.777639


In [4]:
features = ['Horsepower', 'Engine_size', 'Curb_weight', 'Fuel_efficiency', 'Wheelbase', 'Fuel_capacity']
target = 'Price_in_thousands'
df_clean = data[[target] + features].dropna().reset_index(drop=True)

X = df_clean[features]
y = df_clean[target]

## Section2: Objective of the analysis

The goal of this project is to model and predict car prices using vehicle specks and understand which specifications most strongly influence them using linear regression techniques.  By comparing ordinary least squares(OLS) against regularized  methods (L1,L2) and polynomial features through cross validation, the analysis aims to determine the most accurate and statistically stable model.

## Section3: Model Comparison

To predict car prices and examine linear relationships, we formulated five regression setups: baseline Ordinary Least Squares (OLS), Ridge (L2 regularization), Lasso (L1 regularization), and second-degree Polynomial Features combined with both Ridge and Lasso. All models were built inside scikit-learn pipelines with standard feature scaling to prevent data leakage during preprocessing. For Ridge and Lasso variants, regularization penalties (α) were tuned dynamically using internal cross-validation across logarithmic grids. Final model performance was evaluated using 5fold cross validation, recording the R2 score and Root Mean Squared Error (RMSE) to identify the best balance between complexity and predictive accuracy.

In [5]:
#model 1 (OLS)

results = []
s = StandardScaler()
lr = LinearRegression()
pipe1 = Pipeline([('skaler',s),
                 ('regression',lr)])
pred1=cross_val_predict(pipe1,X,y,cv=5)
r2_1=r2_score(y,pred1)
rmse_1 = root_mean_squared_error(y, pred1)
results.append({
    'Model': 'OLS (Baseline)',
    'Best Alpha': '-',
    'CV R2': r2_1,
    'CV RMSE ($k)': rmse_1
})

In [6]:
#model 2 (ridge)

alphas = np.logspace(-3,6,10)
rdg =RidgeCV(alphas=alphas, cv=5)
pipe2 = Pipeline([('skaler',s),
                 ('regression',rdg)])

pred2 = cross_val_predict(pipe2,X,y,cv=5)
r2_2= r2_score(y,pred2)
rmse_2 = root_mean_squared_error(y, pred2)

pipe2.fit(X, y)
best_alpha_ridge = float(pipe2.named_steps['regression'].alpha_)

results.append({
    'Model': 'Ridge',
    'Best Alpha': best_alpha_ridge,
    'CV R2': r2_2,
    'CV RMSE ($k)': rmse_2
})


In [7]:
#model 3 (LASSO)

alphas = np.logspace(-4, 3, 40)
las=LassoCV(alphas=alphas, cv=5, max_iter=10000)
pipe3 = Pipeline([
    ('scaler', s),
    ('regression', las)])

pred3 = cross_val_predict(pipe3, X, y, cv=5)
r2_3 = r2_score(y, pred3)
rmse_3 = root_mean_squared_error(y, pred3)

pipe3.fit(X, y)
best_alpha_lasso = float(pipe3.named_steps['regression'].alpha_)

results.append({
    'Model': 'Lasso (L1)',
    'Best Alpha': best_alpha_lasso, 
    'CV R2': r2_3,
    'CV RMSE ($k)': rmse_3
})

In [8]:
# model 4 Polynomial features , Ridge

alphas = np.logspace(-3,3,7)
rdg =RidgeCV(alphas=alphas, cv=5)
pol = PolynomialFeatures(degree=2,include_bias=False)
pipe4 = Pipeline([('polynomial',pol),
                  ('skaler',s),
                 ('regression',rdg)])

pred4 = cross_val_predict(pipe4,X,y,cv=5)
r2_4= r2_score(y,pred4)
rmse_4 = root_mean_squared_error(y, pred4)

pipe4.fit(X, y)
best_alpha_pf_ridge = float(pipe4.named_steps['regression'].alpha_)

results.append({
    'Model': 'PF + Ridge',
    'Best Alpha': best_alpha_pf_ridge,
    'CV R2': r2_4,
    'CV RMSE ($k)': rmse_4
})

In [9]:
# model 5 (Polynomial Features + LASSO)

alphas = np.logspace(-2, 3, 30)
las=LassoCV(alphas=alphas, cv=5, max_iter=10000)

pipe5 = Pipeline([('polynomial',pol),
    ('scaler', s),
    ('regression', las)])

pred5 = cross_val_predict(pipe5, X, y, cv=5)
r2_5 = r2_score(y, pred5)
rmse_5 = root_mean_squared_error(y, pred5)

pipe5.fit(X, y)
best_alpha_lasso = float(pipe5.named_steps['regression'].alpha_)

results.append({
    'Model': 'PF + Lasso (L1)',
    'Best Alpha': best_alpha_lasso, 
    'CV R2': r2_5,
    'CV RMSE ($k)': rmse_5
})

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.054007e+00, tolerance: 2.331e+00
  model = cd_fast.enet_coordinate_descent_gram(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.631178e+01, tolerance: 2.117e+00
  model = cd_fast.enet_coordinate_descent_gram(
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:827: ConvergenceWarning: Objective did not converge. You might want to increase th

In [10]:
df_results = pd.DataFrame(results)
df_results

,Model,Best Alpha,CV R2,CV RMSE ($k)
0,OLS (Baseline),-,0.714675,7.676474
1,Ridge,1.0,0.718870,7.619832
2,Lasso (L1),0.0001,0.708259,7.762297
3,PF + Ridge,10.0,0.693013,7.962543
4,PF + Lasso (L1),0.022122,0.667037,8.292589


## Section4: Key Findings

As shown in the comparison table, Ridge regression achieved the strongest overall performance, outperforming both the baseline OLS and the polynomial variants. Inspecting the standardized coefficients of the winning Ridge model reveals that Horsepower is by far the most influential positive driver of vehicle price (β≈+14.26), followed by Curb weight (β≈+4.59). Conversely, Engine size exhibits a negative coefficient (β≈−4.69β≈−4.69), which reflects multicollinearity with horsepower indicating that once engine output is accounted for, larger displacement does not independently command a premium.The intercept at 27.33 represents the baseline average price for an average specified vehicle.

## Section5: Limitations and next steps

A key limitation of this project is the modest sample size (approximately 150 instances) and the presence of right-skewed pricing outliers, which constrained more complex polynomial architectures and induced mild overfitting. Future work should explore a logarithmic transformation on vehicle prices log(y) to stabilize variance, expand data collection across broader manufacturing segments, and benchmark against tree-based ensembles (such as Random Forests or Gradient Boosting) to capture non-linear market interactions without manual feature engineering.